# 03 - Pipeline de Preprocesamiento Retiniano (Fases 1-6)

**Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad**

- **Autor:** Jesús Goenaga Peña
- **Programa:** Doctorado en Ciencias Cognitivas - Universidad Autónoma de Manizales

---

Este notebook implementa las **Fases 1-6** del modelo cognitivo artificial,
que simulan el procesamiento visual desde el ingreso de luz en la pupila
hasta la formación de las cintillas ópticas:

| Fase | Proceso Biológico | Operación Computacional |
|------|-------------------|--------------------------|
| 1 | Ingreso de luz a la pupila | Normalización, histograma, suavizado gaussiano |
| 2 | Refracción en córnea y cristalino | Filtro gaussiano + laplaciano |
| 3 | Proyección en la retina | Retinotopía, separación conos/bastones, nasal/temporal |
| 4 | Transformación en impulsos nerviosos | Transducción log, inhibición lateral, mapas on/off |
| 5 | Quiasma óptico | Cruce contralateral de fibras nasales |
| 6 | Cintillas ópticas | Agrupación por hemisferio |

## 1. Configuración

In [ ]:
# Montar Drive y configurar rutas
from google.colab import drive
drive.mount('/content/drive')

import os, sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import glob
from PIL import Image

PROJECT_ROOT = '/content/drive/MyDrive/cognitive-depth-model'
REPO_DIR = '/content/cognitive-depth-model'

# Clonar o actualizar el repositorio
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/yisusgoenaga/cognitive-depth-model.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

print('Configuración lista.')

In [ ]:
# Importar el pipeline retiniano
from preprocessing.retinal_pipeline import (
    phase1_pupil_light_entry,
    phase2_cornea_lens_refraction,
    phase3_retinal_projection,
    phase4_nerve_impulse_transformation,
    phase5_optic_chiasma,
    phase6_optic_tracts,
    run_retinal_pipeline
)

print('Pipeline retiniano importado correctamente.')

In [ ]:
# Cargar un par estereoscópico de KITTI para demostración
KITTI_RAW = os.path.join(PROJECT_ROOT, 'data/raw/kitti')

# Buscar las carpetas de imágenes
def find_kitti_images(base):
    for root, dirs, files in os.walk(base):
        if os.path.basename(root) == 'image_2' and 'training' in root:
            return os.path.dirname(root)
    return None

kitti_training = find_kitti_images(KITTI_RAW)

if kitti_training:
    left_dir = os.path.join(kitti_training, 'image_2')
    right_dir = os.path.join(kitti_training, 'image_3')
    disp_dir = os.path.join(kitti_training, 'disp_occ_0')

    # Cargar la primera escena (emparejar por nombre)
    disp_files = sorted(glob.glob(os.path.join(disp_dir, '*_10.png')))
    sample_id = os.path.basename(disp_files[5]).replace('_10.png', '')  # Escena 5

    img_left = cv2.imread(os.path.join(left_dir, f'{sample_id}_10.png'))
    img_right = cv2.imread(os.path.join(right_dir, f'{sample_id}_10.png'))
    img_disp = cv2.imread(os.path.join(disp_dir, f'{sample_id}_10.png'), cv2.IMREAD_UNCHANGED)

    print(f'Par estereoscópico cargado: escena {sample_id}')
    print(f'  Izquierda: {img_left.shape}')
    print(f'  Derecha:   {img_right.shape}')
    print(f'  Disparidad: {img_disp.shape}')

    # Mostrar el par original
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].imshow(cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Ojo Izquierdo (original)')
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(img_right, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Ojo Derecho (original)')
    axes[1].axis('off')
    axes[2].imshow(img_disp.astype(np.float32) / 256.0, cmap='magma')
    axes[2].set_title('Mapa de Disparidad (ground truth)')
    axes[2].axis('off')
    plt.suptitle(f'Escena KITTI {sample_id} - Entrada al modelo', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('ERROR: No se encontró KITTI. Verifica la ruta en Drive.')

## 2. Fase 1 — Ingreso de Luz a la Pupila

**Proceso biológico:** La pupila regula dinámicamente la cantidad de luz que entra al ojo.

**Operación computacional:**
- Normalización de intensidad: $I_{norm} = \frac{I - \min(I)}{\max(I) - \min(I)}$
- Igualación de histograma (canal de luminancia)
- Filtro gaussiano para reducción de ruido

In [ ]:
# Ejecutar Fase 1
left_p1 = phase1_pupil_light_entry(img_left)
right_p1 = phase1_pupil_light_entry(img_right)

# Visualizar
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Fase 1: Ingreso de Luz a la Pupila', fontsize=14, fontweight='bold')

axes[0, 0].imshow(cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Ojo Izquierdo - Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(cv2.cvtColor((left_p1 * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('Ojo Izquierdo - Fase 1')
axes[0, 1].axis('off')

axes[1, 0].imshow(cv2.cvtColor(img_right, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Ojo Derecho - Original')
axes[1, 0].axis('off')

axes[1, 1].imshow(cv2.cvtColor((right_p1 * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[1, 1].set_title('Ojo Derecho - Fase 1')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results/visualizations/phase1_pupil.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Rango de valores - Original: [{img_left.min()}, {img_left.max()}]')
print(f'Rango de valores - Fase 1:   [{left_p1.min():.3f}, {left_p1.max():.3f}]')

## 3. Fase 2 — Refracción en Córnea y Cristalino

**Proceso biológico:** La córnea suaviza la luz; el cristalino ajusta el enfoque fino.

**Operación computacional:**
- Córnea: filtro gaussiano (suavizado general)
- Cristalino: filtro laplaciano (mejora de nitidez local)

In [ ]:
# Ejecutar Fase 2
left_p2 = phase2_cornea_lens_refraction(left_p1)
right_p2 = phase2_cornea_lens_refraction(right_p1)

# Visualizar comparación Fase 1 vs Fase 2 (detalle ampliado)
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle('Fase 2: Refracción en Córnea y Cristalino', fontsize=14, fontweight='bold')

# Recorte central para ver diferencias de enfoque
h, w = left_p1.shape[:2]
crop = (slice(h//3, 2*h//3), slice(w//3, 2*w//3))

axes[0, 0].imshow(cv2.cvtColor((left_p1[crop] * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Fase 1 (detalle central)')
axes[0, 0].axis('off')

axes[0, 1].imshow(cv2.cvtColor((left_p2[crop] * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('Fase 2 (detalle central) - Enfoque mejorado')
axes[0, 1].axis('off')

# Imagen completa Fase 2
axes[1, 0].imshow(cv2.cvtColor((left_p2 * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[1, 0].set_title('Ojo Izquierdo - Fase 2 completa')
axes[1, 0].axis('off')

axes[1, 1].imshow(cv2.cvtColor((right_p2 * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[1, 1].set_title('Ojo Derecho - Fase 2 completa')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results/visualizations/phase2_refraction.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Fase 3 — Proyección en la Retina

**Proceso biológico:** La retina tiene densidad variable de receptores (fóvea vs periferia)
y dos tipos de fotorreceptores: conos (color/detalle) y bastones (luminancia/movimiento).

**Operación computacional:**
- Transformación retinotópica con máscara foveal gaussiana
- Separación en canal de conos y canal de bastones
- División en regiones nasal y temporal

In [ ]:
# Ejecutar Fase 3
left_p3 = phase3_retinal_projection(left_p2)
right_p3 = phase3_retinal_projection(right_p2)

# Visualizar los 4 canales del ojo izquierdo
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Fase 3: Proyección Retiniana - Ojo Izquierdo', fontsize=14, fontweight='bold')

# Conos nasal
cones_n = left_p3['cones_nasal']
if len(cones_n.shape) == 3:
    axes[0, 0].imshow(cv2.cvtColor((cones_n * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
else:
    axes[0, 0].imshow(cones_n, cmap='gray')
axes[0, 0].set_title('Conos - Región Nasal\n(color y detalle)')
axes[0, 0].axis('off')

# Conos temporal
cones_t = left_p3['cones_temporal']
if len(cones_t.shape) == 3:
    axes[0, 1].imshow(cv2.cvtColor((cones_t * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
else:
    axes[0, 1].imshow(cones_t, cmap='gray')
axes[0, 1].set_title('Conos - Región Temporal\n(color y detalle)')
axes[0, 1].axis('off')

# Bastones nasal
axes[1, 0].imshow(left_p3['rods_nasal'], cmap='gray')
axes[1, 0].set_title('Bastones - Región Nasal\n(luminancia y movimiento)')
axes[1, 0].axis('off')

# Bastones temporal
axes[1, 1].imshow(left_p3['rods_temporal'], cmap='gray')
axes[1, 1].set_title('Bastones - Región Temporal\n(luminancia y movimiento)')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results/visualizations/phase3_retina.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Canales generados por ojo: {len(left_p3)} (conos nasal/temporal + bastones nasal/temporal)')

## 5. Fase 4 — Transformación en Impulsos Nerviosos

**Proceso biológico:** Las células ganglionares convierten la señal luminosa en
impulsos nerviosos, con patrones de campo receptivo on-center y off-center.

**Operación computacional:**
- Transducción visual: $R = \log(1 + I)$
- Inhibición lateral: $I_{inh}(x) = I(x) - \lambda \sum_{y \in N(x)} I(y)$
- Mapas ganglionares on-center (centro excitatorio) y off-center (periferia excitatoria)

In [ ]:
# Ejecutar Fase 4
left_p4 = phase4_nerve_impulse_transformation(left_p3)
right_p4 = phase4_nerve_impulse_transformation(right_p3)

# Visualizar mapas ganglionares del ojo izquierdo (conos nasales como ejemplo)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Fase 4: Mapas Ganglionares - Ojo Izquierdo', fontsize=14, fontweight='bold')

# On-center conos
on_map = left_p4['cones_nasal_on']
if len(on_map.shape) == 3:
    on_display = np.mean(on_map, axis=2)
else:
    on_display = on_map
axes[0, 0].imshow(on_display, cmap='hot')
axes[0, 0].set_title('Conos Nasal - ON-center\n(centro excitatorio)')
axes[0, 0].axis('off')

# Off-center conos
off_map = left_p4['cones_nasal_off']
if len(off_map.shape) == 3:
    off_display = np.mean(off_map, axis=2)
else:
    off_display = off_map
axes[0, 1].imshow(off_display, cmap='hot')
axes[0, 1].set_title('Conos Nasal - OFF-center\n(periferia excitatoria)')
axes[0, 1].axis('off')

# On-center bastones
axes[1, 0].imshow(left_p4['rods_nasal_on'], cmap='hot')
axes[1, 0].set_title('Bastones Nasal - ON-center')
axes[1, 0].axis('off')

# Off-center bastones
axes[1, 1].imshow(left_p4['rods_nasal_off'], cmap='hot')
axes[1, 1].set_title('Bastones Nasal - OFF-center')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results/visualizations/phase4_ganglion.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Mapas ganglionares por ojo: {len(left_p4)}')
print(f'Tipos: {list(left_p4.keys())}')

## 6. Fase 5 — Quiasma Óptico

**Proceso biológico:** Las fibras nasales cruzan al hemisferio opuesto (contralateral),
mientras las temporales permanecen ipsilaterales.

**Resultado:** Cada hemisferio recibe información del campo visual contralateral completo.

In [ ]:
# Ejecutar Fase 5
hemispheres = phase5_optic_chiasma(left_p4, right_p4)

# Verificar el cruce
print('Fase 5: Quiasma Óptico - Verificación del cruce de fibras')
print('=' * 60)
print()
print('Hemisferio IZQUIERDO recibe:')
print('  - Temporal del ojo izquierdo (ipsilateral)')
print('  - Nasal del ojo derecho (contralateral)')
print(f'  -> Canales: {list(hemispheres["left_hemisphere"].keys())}')
print(f'  -> Señales por canal: {len(hemispheres["left_hemisphere"]["cones_on"])}')
print()
print('Hemisferio DERECHO recibe:')
print('  - Nasal del ojo izquierdo (contralateral)')
print('  - Temporal del ojo derecho (ipsilateral)')
print(f'  -> Canales: {list(hemispheres["right_hemisphere"].keys())}')
print(f'  -> Señales por canal: {len(hemispheres["right_hemisphere"]["cones_on"])}')
print()
print('Cruce verificado correctamente.')

## 7. Fase 6 — Cintillas Ópticas

**Proceso biológico:** Las señales se agrupan en cintillas ópticas izquierda y derecha
para su transmisión hacia el NGL.

**Operación computacional:** Reorganización estructural sin procesamiento adicional.

In [ ]:
# Ejecutar Fase 6
optic_tracts = phase6_optic_tracts(hemispheres)

# Resumen de la estructura final
print('Fase 6: Cintillas Ópticas - Estructura final')
print('=' * 60)

for tract_name, tract_data in optic_tracts.items():
    print(f'\n{tract_name.upper().replace("_", " ")}:')
    for receptor_type, signals in tract_data.items():
        for response_type, signal_list in signals.items():
            shapes = [s.shape for s in signal_list]
            print(f'  {receptor_type} [{response_type}]: {len(signal_list)} señales, formas: {shapes}')

## 8. Pipeline Completo — Ejecución Integrada

In [ ]:
# Ejecutar el pipeline completo en una sola llamada
results = run_retinal_pipeline(img_left, img_right)

print('Pipeline retiniano ejecutado exitosamente.')
print(f'Fases completadas: {[k for k in results.keys() if k.startswith("phase")]}')
print()

# Resumen visual: una imagen por fase
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Resumen del Pipeline Retiniano (Fases 1-6) - Ojo Izquierdo',
             fontsize=14, fontweight='bold')

# Fase 1
p1 = results['phase1']['left']
axes[0, 0].imshow(cv2.cvtColor((p1 * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Fase 1: Pupila\n(normalización + histograma)')
axes[0, 0].axis('off')

# Fase 2
p2 = results['phase2']['left']
axes[0, 1].imshow(cv2.cvtColor((p2 * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
axes[0, 1].set_title('Fase 2: Córnea + Cristalino\n(enfoque óptico)')
axes[0, 1].axis('off')

# Fase 3: conos
p3 = results['phase3']['left']
cones_full = np.concatenate([p3['cones_nasal'], p3['cones_temporal']], axis=1)
if len(cones_full.shape) == 3:
    axes[0, 2].imshow(cv2.cvtColor((cones_full * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
else:
    axes[0, 2].imshow(cones_full, cmap='gray')
axes[0, 2].set_title('Fase 3: Retina (conos)\n(nasal | temporal)')
axes[0, 2].axvline(x=cones_full.shape[1]//2, color='red', linewidth=1, linestyle='--')
axes[0, 2].axis('off')

# Fase 3: bastones
rods_full = np.concatenate([p3['rods_nasal'], p3['rods_temporal']], axis=1)
axes[1, 0].imshow(rods_full, cmap='gray')
axes[1, 0].set_title('Fase 3: Retina (bastones)\n(nasal | temporal)')
axes[1, 0].axvline(x=rods_full.shape[1]//2, color='red', linewidth=1, linestyle='--')
axes[1, 0].axis('off')

# Fase 4: ganglionares on-center
p4 = results['phase4']['left']
on_map = p4['cones_nasal_on']
if len(on_map.shape) == 3:
    on_map = np.mean(on_map, axis=2)
axes[1, 1].imshow(on_map, cmap='hot')
axes[1, 1].set_title('Fase 4: Ganglionares\n(on-center, conos nasal)')
axes[1, 1].axis('off')

# Fase 5-6: esquema de cruce
axes[1, 2].text(0.5, 0.7, 'Fase 5: Quiasma Óptico', ha='center', fontsize=12, fontweight='bold',
                transform=axes[1, 2].transAxes)
axes[1, 2].text(0.5, 0.55, 'Nasal → Contralateral', ha='center', fontsize=10,
                transform=axes[1, 2].transAxes)
axes[1, 2].text(0.5, 0.42, 'Temporal → Ipsilateral', ha='center', fontsize=10,
                transform=axes[1, 2].transAxes)
axes[1, 2].text(0.5, 0.2, 'Fase 6: Cintillas Ópticas', ha='center', fontsize=12, fontweight='bold',
                transform=axes[1, 2].transAxes)
axes[1, 2].text(0.5, 0.05, '→ Listas para el NGL (Fase 7)', ha='center', fontsize=10, style='italic',
                transform=axes[1, 2].transAxes)
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'results/visualizations/pipeline_summary_phases1to6.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Resumen guardado en results/visualizations/pipeline_summary_phases1to6.png')

In [ ]:
print('=' * 60)
print('NOTEBOOK 03 COMPLETADO')
print('=' * 60)
print()
print('Fases implementadas: 1-6 (Pipeline Retiniano)')
print()
print('Módulo creado: src/preprocessing/retinal_pipeline.py')
print('  - phase1_pupil_light_entry()')
print('  - phase2_cornea_lens_refraction()')
print('  - phase3_retinal_projection()')
print('  - phase4_nerve_impulse_transformation()')
print('  - phase5_optic_chiasma()')
print('  - phase6_optic_tracts()')
print('  - run_retinal_pipeline()  <-- Pipeline completo')
print()
print('Siguiente: Notebook 04 - Arquitectura del Modelo (Fases 7-27)')
print('  NGL, V1, V2, V3/V3A, V4, V5/MT como arquitectura ResNet unificada')